# 🧠 SarvGyan Backend Test Environment
---
This isolated notebook allows you to test the core RAG components (LLM, Embeddings, ChromaDB, QA Chain) independently, completely bypassing the Streamlit frontend. It is excluded from Git via `.gitignore`.

In [4]:
import os
import sys
from pathlib import Path

# Ensure project root is in the python path
project_root = str(Path().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))
print("Environment variables loaded.")

Environment variables loaded.


## 1. Test LLM (Groq API)

In [5]:
from src.llm_handler import generate_response

prompt = "Hello, are you receiving this message? Reply with only 'Yes, systems are nominal.'"
response = generate_response(prompt)
print(f"LLM Output: {response}")

LLM Output: Yes, systems are nominal.


## 2. Test Embeddings Engine
This tests the local `sentence-transformers` model (`BAAI/bge-large-en-v1.5`).

In [6]:
from src.embeddings import generate_single_embedding

test_text = "This is a test document block for embedding generation."
embedding = generate_single_embedding(test_text)
print(f"Generated embedding vector. Dimension size: {len(embedding)} (Expected: 1024)")

Generated embedding vector. Dimension size: 1024 (Expected: 1024)


## 3. Test Vector Store & QA Chain
This connects to ChromaDB and initializes the RAG pipeline orchestrator.

In [7]:
from src.qa_chain import QAChain

qa = QAChain()
docs = qa.get_indexed_documents()
print(f"Currently indexed documents in persistent DB: {docs}")

Currently indexed documents in persistent DB: []


## 4. Run Raw Semantic Search
Use this cell to see exactly what context chunks the LLM sees when a user asks a question.

In [9]:
query = "What is the main topic of the uploaded documents?"
num_results = 2

if docs:
    results = qa.vector_store.search(query, n_results=num_results)
    print(f"RAW CONTEXT CHUNKS RETRIEVED (Top {num_results}):")
    for i, r in enumerate(results):
        print(f"\n--- Chunk {i+1} ---")
        print(f"Source: {r['metadata'].get('source')}")
        snippet = r['text'][:300] + "..." if len(r['text']) > 300 else r['text']
        print(f"Text:\n{snippet}")
else:
    print("No documents are currently indexed to search. Go to Streamlit and upload one first.")

No documents are currently indexed to search. Go to Streamlit and upload one first.


## 5. DANGER ZONE: Wipe Database
This will completely destroy your local ChromaDB collection.

In [ ]:
# UNCOMMENT THE NEXT LINE TO WIPE ALL CHROMADB DATA AND UPLOADS
# qa.vector_store.reset()
# print("Database wiped entirely.")